# Luma using DTL workflow demonstration for Pagar Alam area

In [ ]:
!python -m pip install .. --quiet

In [ ]:
import ee 

ee.Authenticate() 
ee.Initialize()

# 1. Satellite imagery

In [ ]:
import geemap
from luma_ge.data_acquisition import Reflectance_Data, final_Image

# AOI definition

aoi = geemap.shp_to_ee('../data/modular_mapping_approach/pagaralam_test/Kota_Pagar_Alam.shp')

#========== FIRST RETRIVE THE MULTISPECTRAL BAND===========
#Intialize the relfectance class data function
optical_reflectance = Reflectance_Data()
#Initialize the final image class for composite creation
composite = final_Image() #NEW FEATURE ADDED HERE
#define the start and end date for imagery collection
start = '2024-01-01'
end = '2024-12-31'
#get the image collection and corresponding statistics
landsat_data, meta = optical_reflectance.get_optical_data(aoi, start, end, optical_data='L8_SR', 
                                                           cloud_cover=40, compute_detailed_stats=False)
#create mosaic between image collection, and clip based on AOI
mosaic_landsat = composite.get_quality_mosaic(landsat_data, aoi, quality_band= 'NDVI', calculate_coverage=False) #REPLACE OLD CODE WITH THE NEW ONE HERE
#Alternatively you can use temporal aggregation (ee reducer) to create mode cloudless imagery
#Add new functionality to calculate the coverage of the composite
median_landsat, coverage = composite.get_temporal_composite(landsat_data, aoi, reducer='Median', calculate_coverage=True) #REPLACE OLD CODE WITH THE NEW ONE HERE
#visualization parameter
l8_sr_visparam = {'min': 0,'max': 0.4,'gamma': [0.95, 1.1, 1],'bands':['NIR', 'RED', 'GREEN']}

#retive thermal bands from TOA
thermal_bands, thermal_stats = optical_reflectance.get_thermal_bands(aoi, start, end, cloud_cover=40, thermal_data='L8_TOA', compute_detailed_stats=False)
median_thermal = composite.get_temporal_composite(thermal_bands, aoi, reducer='Median') #REPLACE THE OLD CODE WITH THE NEW ONE
thermal_vis = {'min': 286,'max': 300,'gammma': 0.4}
#stacked all landsat bands and convert to float(making sure all data type are compatible)
stacked_landsat = median_landsat.addBands(median_thermal).toFloat()


# 2. Classification scheme 

In [ ]:
from luma_ge.classification_scheme import LULC_Scheme_Manager

manager = LULC_Scheme_Manager()

scheme_name = "Epistem"
success, message = manager.load_default_scheme(scheme_name)
classification_df = manager.get_dataframe()

print(classification_df.to_string(index=False))

# 3. Upload modular reference data

In [ ]:
import pandas as pd
import numpy as np

# use the modular reference dataset was reverse engineered from KLHK map
df_train_csv = '../data/modular_mapping_approach/pagaralam_test/pagaralam_test_revengineer2.csv'
df_train = pd.read_csv(df_train_csv)

# Extract lon and lat
df_train[['longitude', 'latitude']] = (
    df_train['geometry']
    .str.extract(r'POINT\s*\(([-\d.]+)\s+([-\d.]+)\)')
    .astype(float)
)

df_train = df_train.drop(columns=['geometry'])

print(df_train.head())

## Define default scheme labelling ruleset

In [ ]:
# ── DEFINE RULESET ─────────────────────────────────────────────────────────
# Each row = one class rule. Columns are primitives with operators.
# Format: ">0.40" means "greater than 0.40"
#         "==1" means "equal to 1"
#         ">=25" means "greater than or equal to 25"
#         'treecover': '<30 | >80',  means treecover < 30 OR treecover > 80   
#         None means "no condition on this primitive"

ruleset_csv = '../data/modular_mapping_approach/ruleset_epistem_default.csv'
ruleset = pd.read_csv(ruleset_csv)
ruleset = ruleset.sort_values(by='priority', ascending=True)
print(ruleset)

# copy to clipboard for easy pasting into the ruleset CSV file
pd.DataFrame.to_clipboard(ruleset)

## Helper functions to label the classes

In [ ]:
# ── HELPER FUNCTIONS ───────────────────────────────────────────────────────

def safe_num(val, default=0):
    """Convert value to float, return default if None or NaN."""
    if val is None:
        return default
    if isinstance(val, (int, float)):
        if np.isnan(val):
            return default
        return float(val)
    try:
        return float(val)
    except:
        return default

def evaluate_condition(row_val, condition_str):
    """
    Evaluate a single condition: row_val op threshold?
    Supports OR logic with pipe separator: ">0.40|<0.10"
    
    Args:
        row_val: The value from the sample
        condition_str: String like ">0.40", ">=25", ">0.40|<0.10"
    
    Returns:
        bool: True if condition is satisfied, False otherwise
    """
    if condition_str is None:
        return True  # No condition → always passes
    
    condition_str = str(condition_str).strip()
    
    # Handle OR logic (pipe-separated conditions)
    if '|' in condition_str:
        sub_conditions = [c.strip() for c in condition_str.split('|')]
        return any(evaluate_condition(row_val, c) for c in sub_conditions)
    
    # Parse operator and threshold
    if condition_str.startswith('=='):
        op, threshold_str = '==', condition_str[2:]
    elif condition_str.startswith('>='):
        op, threshold_str = '>=', condition_str[2:]
    elif condition_str.startswith('<='):
        op, threshold_str = '<=', condition_str[2:]
    elif condition_str.startswith('>'):
        op, threshold_str = '>', condition_str[1:]
    elif condition_str.startswith('<'):
        op, threshold_str = '<', condition_str[1:]
    else:
        return True  # Invalid condition → pass
    
    threshold = safe_num(threshold_str)
    row_val_num = safe_num(row_val)
    
    if op == '>':
        return row_val_num > threshold
    elif op == '>=':
        return row_val_num >= threshold
    elif op == '<':
        return row_val_num < threshold
    elif op == '<=':
        return row_val_num <= threshold
    elif op == '==':
        return row_val_num == threshold
    
    return False

def check_rule_match(row, rule):
    """
    Check if a sample row matches all conditions in a rule.
    
    Args:
        row: pd.Series with sample data
        rule: pd.Series with rule conditions
    
    Returns:
        bool: True if ALL conditions are satisfied
    """
    # Get all primitive columns (skip metadata like class_id, class_name, priority)
    metadata = {'class_id', 'class_name', 'priority'}
    primitive_cols = [col for col in rule.index if col not in metadata and col != 'treecover_max']
    
    for prim in primitive_cols:
        condition = rule[prim]
        
        # Handle special case for range checks (e.g., treecover_max)
        if pd.isna(condition) or condition is None:
            continue  # No condition on this primitive
        
        row_val = row.get(prim, np.nan)
        
        if not evaluate_condition(row_val, condition):
            return False  # Any condition fails → rule doesn't match
    
    return True  # All conditions passed

print('✓ Helper functions defined')

## Assign the class labels to the modular reference data

In [ ]:
# ── ASSIGN LABELS VIA DECISION RULES ───────────────────────────────────────

def assign_label(row, ruleset):
    """
    Evaluate all rules in priority order. First match → assign that class.
    If no match → return 0 (unclassified).
    
    This mirrors GEE's nested ee.Algorithms.If logic:
    Each rule's NO branch continues to the next rule.
    First YES match stops evaluation.
    """
    # Sort by priority to ensure correct evaluation order
    ruleset_sorted = ruleset.sort_values('priority').reset_index(drop=True)
    
    for _, rule in ruleset_sorted.iterrows():
        if check_rule_match(row, rule):
            return rule['class_id']
    
    return 0  # No rule matched → unclassified

# Apply to all training samples
print('[Step 3] Assigning labels via decision rules...')

# # IF only use a subset of a class, then apply this instead:
# ruleset_subset = ruleset[ruleset['class_id'].isin([23, 20, 13])]  # Only 3 classes

df_train['label'] = df_train.apply(lambda row: assign_label(row, ruleset), axis=1)
# ruleset_subset = ruleset[ruleset['class_id'].isin([2, 4, 9, 13, 18, 19, 20, 21, 23])]
# df_split_train['label'] = df_split_train.apply(lambda row: assign_label(row, ruleset_subset), axis=1)

# add class_name column
class_mapping = (
    ruleset[['class_id', 'class_name']]
    .drop_duplicates()
    .set_index('class_id')['class_name']
    .to_dict()
)
df_train['class_name'] = df_train['label'].map(class_mapping)
# Handle unclassified
df_train['class_name'] = df_train['class_name'].fillna('unclassified')

print(f'✓ Labels assigned to {len(df_train)} samples\n')
print('Label distribution (assigned):')
print(df_train['class_name'].value_counts().sort_index())
print(f'\nUnclassified (label=0): {(df_train["label"] == 0).sum()}')

## Compare with original class label

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ORIGINAL_CLASS = "LULC_24"

print(f"Classified samples: {len(df_train)}")
print()

# Count how many samples from each original class received each label
summary = pd.crosstab(
    df_train[ORIGINAL_CLASS],
    df_train["class_name"]
)

print("Original class -> Assigned labels")
print(summary)

summary.to_clipboard(
    excel=True,
    index=True
)

## Save the labelled training dataset

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

geometry = [Point(xy) for xy in zip(df_train["longitude"], df_train["latitude"])]
gdf = gpd.GeoDataFrame(df_train, geometry=geometry)
gdf.to_file("../data/temp/pagaralam_relabelled.shp")

## Sanitize relabelled shp file with original Luma module 3 script

In [ ]:
from luma_ge.sample_data import SyncTrainData

LULCTable = classification_df
TrainVectPath = "../data/temp/pagaralam_relabelled.shp"

TrainField = 'label' 
        # Load and process training data
TrainDataDict = SyncTrainData.LoadTrainData(
            landcover_df=LULCTable,
            aoi_geometry=aoi,
            training_shp_path=TrainVectPath
        )

In [ ]:
# ----- System response 3.2.a -----
# Set class field
TrainDataDict = SyncTrainData.SetClassField(TrainDataDict, TrainField)

# Validate classes
TrainDataDict = SyncTrainData.ValidClass(TrainDataDict, 1)

    # Check sample sufficiency
TrainDataDict = SyncTrainData.CheckSufficiency(TrainDataDict, min_samples=20)

    # Filter by AOI
TrainDataDict = SyncTrainData.FilterTrainAoi(TrainDataDict)

    # Create training data table
table_df, total_samples, insufficient_df = SyncTrainData.TrainDataRaw(
    training_data=TrainDataDict.get('training_data'),
    landcover_df=TrainDataDict.get('landcover_df'),
    class_field=TrainDataDict.get('class_field'))

#Summary result
vr = TrainDataDict.get('validation_results', {})

print("=" * 70)
print("TRAINING DATA SUMMARY")
print("=" * 70)
print(f"Total training points loaded     : {vr.get('total_points', 'N/A')}")
print(f"Points after class filtering     : {vr.get('points_after_class_filter', 'N/A')}")
print(f"Valid points (inside AOI)        : {vr.get('valid_points', 'N/A')}")
print(f"Invalid classes found            : {len(vr.get('invalid_classes', []))}")
print(f"Points outside AOI               : {len(vr.get('outside_aoi', []))}")
print("=" * 70)

    # --- Display the main table ---
if table_df is not None and not table_df.empty:
        display_df = table_df.copy()
        if 'Percentage' in display_df.columns:
            display_df['Percentage'] = display_df['Percentage'].apply(
                lambda x: f"{x:.2f}%" if isinstance(x, (int, float)) else x
            )
        display(display_df)
else:
        print("No valid training data available to display.")

TrainDataFinal = TrainDataDict.get('training_data')

# 6. Land cover classification

In [ ]:
from luma_ge.classification import FeatureExtraction, Generate_LULC

In [ ]:
labeled_roi = geemap.gdf_to_ee(TrainDataFinal)

#Perform Training Test Split
features = FeatureExtraction()
strafied_train, stratified_test = features.stratified_split(labeled_roi, stacked_landsat, 
                            class_prop='label', train_ratio=1)


In [ ]:
classifier = Generate_LULC()
print("Performing Classification...")
#Multiclass hard classification
classification_map, trained_model = classifier.hard_classification(strafied_train, class_property='label', image=stacked_landsat,
                                                          ntrees=300, min_leaf=2, return_model=True)

In [ ]:
orig_hist = classification_map.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=aoi,
    scale=30,
    maxPixels=1e13
).getInfo()

hist = orig_hist['classification']  

labels = list(hist.keys())
values = list(hist.values())
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.bar(labels, values)

plt.xlabel("Class ID")
plt.ylabel("Pixel Count")
plt.title("Land Cover Class Distribution")
plt.xticks(rotation=45)

plt.show()

## Save classified map

In [ ]:
# task = ee.batch.Export.image.toDrive(
#     image=classification_map,
#     description='pagar alam epistem classification',
#     fileNamePrefix='pagaralam_epistem_classification',
#     region=aoi.geometry(),
#     scale=30,
#     maxPixels=1e13,
#     fileFormat='GeoTIFF'
# )

# task.start()

# print("Export started:", task.id)